In [1]:
# 0) Import packages
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import HistGradientBoostingClassifier

# 1) Load data
TRAIN_PATH = "/Users/christinewu/Desktop/academic resources/Advanced Machine Learning/Homework Files/Homework1- Kaggle Competition/CSV Files/train.csv"
TEST_PATH = "/Users/christinewu/Desktop/academic resources/Advanced Machine Learning/Homework Files/Homework1- Kaggle Competition/CSV Files/test.csv"
SAMPLE_SUB_PATH = "/Users/christinewu/Desktop/academic resources/Advanced Machine Learning/Homework Files/Homework1- Kaggle Competition/CSV Files/sample_submission.csv"

TARGET = "Irrigation_Need"
ID_COL = "id"
RANDOM_STATE = 42
CV_FOLDS = 5
N_JOBS = -1

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

In [2]:
# 2) Split features / target
X = train_df.drop(columns=[TARGET])
y = train_df[TARGET].copy()
X = X.drop(columns=[ID_COL], errors="ignore")
X_test = test_df.drop(columns=[ID_COL], errors="ignore")

# Encode target
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [3]:
# 3) Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

In [4]:
# 4) Prepare data for HistGradientBoosting
#    - numeric: fill with median
#    - categorical: convert to integer category codes consistently
def prepare_hgb_data(X_train_raw: pd.DataFrame, X_test_raw: pd.DataFrame):
    X_train = X_train_raw.copy()
    X_test = X_test_raw.copy()

    # Fill numeric columns
    for col in numeric_features:
        med = X_train[col].median()
        X_train[col] = X_train[col].fillna(med)
        X_test[col] = X_test[col].fillna(med)

    # Encode categorical columns consistently across train/test
    for col in categorical_features:
        train_col = X_train[col].astype("string").fillna("MissingCategory")
        test_col = X_test[col].astype("string").fillna("MissingCategory")

        all_vals = pd.Index(train_col.unique()).union(pd.Index(test_col.unique()))
        mapping = {val: i for i, val in enumerate(all_vals)}

        X_train[col] = train_col.map(mapping).astype("int32")
        X_test[col] = test_col.map(mapping).astype("int32")

    return X_train, X_test

X_prepared, X_test_prepared = prepare_hgb_data(X, X_test)

categorical_indices = [X_prepared.columns.get_loc(col) for col in categorical_features]

In [5]:
# 5) HistGradientBoosting model + tuning
hgb = HistGradientBoostingClassifier(
    random_state=RANDOM_STATE,
    early_stopping=True,
    categorical_features=categorical_indices
)

param_dist = {
    "learning_rate": [0.03, 0.05, 0.08, 0.1],
    "max_iter": [150, 250, 350],
    "max_leaf_nodes": [15, 31, 63],
    "max_depth": [None, 6, 10],
    "min_samples_leaf": [20, 30, 50],
    "l2_regularization": [0.0, 0.1, 1.0],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    estimator=hgb,
    param_distributions=param_dist,
    n_iter=4,
    scoring="accuracy",
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True
)

In [6]:
# 6) Train
search.fit(X_prepared, y_encoded)

print("Best HistGradientBoosting params:")
print(search.best_params_)
print("Best CV accuracy:", round(search.best_score_, 5))

best_model = search.best_estimator_

Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best HistGradientBoosting params:
{'min_samples_leaf': 50, 'max_leaf_nodes': 31, 'max_iter': 350, 'max_depth': 6, 'learning_rate': 0.08, 'l2_regularization': 1.0}
Best CV accuracy: 0.98429


In [7]:
# 7) Fit on full train and predict test
best_model.fit(X_prepared, y_encoded)
test_pred_encoded = best_model.predict(X_test_prepared)
test_pred = label_encoder.inverse_transform(test_pred_encoded)

In [8]:
# 8) Save submission
submission = sample_sub.copy()
submission[TARGET] = test_pred
submission.to_csv("submission_boosting.csv", index=False)